In [1]:
import os
from huggingface_hub import hf_hub_download
from gliner import GLiNER

/home/kojo/Code/ByItsCover/bic-library-search/explore/glenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
repo_id = "urchade/gliner_base"
filenames = ["pytorch_model.bin", "gliner_config.json"]
destination = "search_models_1"

os.makedirs(destination, exist_ok=True)

for file in filenames:
    hf_hub_download(
        repo_id=repo_id,
        filename=file,
        local_dir=destination
    )

In [32]:
extractor = GLiNER.from_pretrained(destination, load_tokenizer=True)
extractor

UniEncoderSpanGLiNER(
  (model): UniEncoderSpanModel(
    (token_rep_layer): Encoder(
      (bert_layer): Transformer(
        (model): DebertaV2Model(
          (embeddings): DebertaV2Embeddings(
            (word_embeddings): Embedding(128004, 768, padding_idx=0)
            (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (encoder): DebertaV2Encoder(
            (layer): ModuleList(
              (0-11): 12 x DebertaV2Layer(
                (attention): DebertaV2Attention(
                  (self): DisentangledSelfAttention(
                    (query_proj): Linear(in_features=768, out_features=768, bias=True)
                    (key_proj): Linear(in_features=768, out_features=768, bias=True)
                    (value_proj): Linear(in_features=768, out_features=768, bias=True)
                    (pos_dropout): Dropout(p=0.1, inplace=False)
                    (dropout): Dropout(p=0.1, inp

In [33]:
text = """
Cristiano Ronaldo dos Santos Aveiro (Portuguese pronunciation: [kɾiʃˈtjɐnu ʁɔˈnaldu]; born 5 February 1985) is a Portuguese professional footballer who plays as a forward for and captains both Saudi Pro League club Al Nassr and the Portugal national team. Widely regarded as one of the greatest players of all time, Ronaldo has won five Ballon d'Or awards,[note 3] a record three UEFA Men's Player of the Year Awards, and four European Golden Shoes, the most by a European player. He has won 33 trophies in his career, including seven league titles, five UEFA Champions Leagues, the UEFA European Championship and the UEFA Nations League. Ronaldo holds the records for most appearances (183), goals (140) and assists (42) in the Champions League, goals in the European Championship (14), international goals (128) and international appearances (205). He is one of the few players to have made over 1,200 professional career appearances, the most by an outfield player, and has scored over 850 official senior career goals for club and country, making him the top goalscorer of all time.
"""

labels = ["person", "award", "date", "competitions", "teams"]

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Cristiano Ronaldo dos Santos Aveiro => person
5 February 1985 => date
Al Nassr => teams
Portugal national team => teams
Ballon d'Or => award
UEFA Men's Player of the Year Awards => award
European Golden Shoes => award
UEFA Champions Leagues => competitions
UEFA European Championship => competitions
UEFA Nations League => competitions
Champions League => competitions
European Championship => competitions


In [34]:
labels = ["Author", "Book_Title_Only", "Genre", "Keywords"]

In [35]:
text = "Brandon Sanderson series with a girl with long hair that takes place in Space"

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Brandon Sanderson => Author
Space => Genre


In [36]:
text = "book where there was a sword on the cover, but it looked kind of old and had oldish lettering. It had characters with powers too"

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

characters => Keywords
powers => Keywords


In [37]:
text = "That one Defensive Baking series, I think by Kingfisher something"

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Defensive Baking => Book_Title_Only
Kingfisher => Author


In [38]:
text = "Fantasy book with dragons in it by Brandom Mull I think"

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Fantasy => Genre
dragons => Keywords
Brandom Mull => Author


In [39]:
text = "A Wizard's Guide To Defensive Baking"

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

A Wizard's Guide To Defensive Baking => Book_Title_Only


In [40]:
text = "there were dragons in it and I think it was by Brandom Mull"

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

dragons => Genre
Brandom Mull => Author


In [41]:
text = "Garth Nix's Keys to the Kingdom series"

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Garth Nix => Author
Keys to the Kingdom => Book_Title_Only


In [42]:
text = (
    "Looking for a Stephen King book. "
    "I don’t remember a lot about it. It was about this white male authour who lives in Maine."
)

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Stephen King => Author


In [44]:
import torch

save_dir = "./search_models_1/onnx"
file_name = "glinner_encoder.onnx"

# input_tensor = torch.ones((2, 3, 224, 224), dtype=torch.float32)

extractor.export_to_onnx(save_dir=save_dir,
                         onnx_filename=file_name
                         )

# torch.onnx.export(extractor.encoder,
#                 (input_tensor),
#                 output_path,
#                 input_names = ['images'],
#                 output_names = ['embeddings'],
#                 dynamic_shapes=({0: torch.export.Dim.DYNAMIC},),
#                 external_data=False
#                 )

/home/kojo/Code/ByItsCover/bic-library-search/explore/glenv/lib/python3.12/site-packages/gliner/modeling/base.py:105: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if target_len == L:
/home/kojo/Code/ByItsCover/bic-library-search/explore/glenv/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 19 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)
[W516 19:08:14.122784636 shape_type_inference.cpp:1994] Warning: The shape inference of prim::PackPadded type is missing, so it may 

{'onnx_path': 'search_models_1/onnx/glinner_encoder.onnx',
 'quantized_path': None}